# Hugging Face Datasets

- [Guide](https://huggingface.co/docs/datasets/how_to)
- [Tutorial](https://huggingface.co/docs/datasets/tutorial)

### Install Dependencies

In [ ]:
!pip install colorgram.py

### Import Libraries

In [ ]:
import colorgram
import json
import webcolors

from datasets import load_dataset
from PIL import ImageDraw as PImageDraw

## Intro to `Dataset`

Slightly fancier version of a `json` file or a `DataFrame`.

For example, we can store and manipulate images inside Hugging Face Datasets.

### Download/Open Dataset

[https://huggingface.co/datasets/thiagohersan/newsdata-images](https://huggingface.co/datasets/thiagohersan/newsdata-images)

In [ ]:
news_ds = load_dataset("thiagohersan/newsdata-images")
display(news_ds)

In [ ]:
# looking at "splits"
news_ds["newsdata"]

In [ ]:
# looking at features
news_ds["newsdata"].features

In [ ]:
# looking at a single row/record
news_ds["newsdata"][0]

A `Dataset` is (mostly) a list... sort of...

In [ ]:
# iterate over entire dataset -> list
for n in news_ds["newsdata"]:
  print(n["source_name"], ":", n["title"][:32], "...")

In [ ]:
# take() selects a subset of the Dataset -> list
display(news_ds["newsdata"].take(3))

for n in news_ds["newsdata"].take(3):
  print(n["title"])
  display(n["image"])

In [ ]:
# select() selects a subset of the Dataset -> list
display(news_ds["newsdata"].select([0,2,4]))

for n in news_ds["newsdata"].select([0,2,4]):
  print(n["title"])
  display(n["image"])

In [ ]:
# if slicing -> dictionary of lists
display(news_ds["newsdata"][:2])

for img in news_ds["newsdata"][:3]["image"]:
  display(img)

### Convert to `json`

In [ ]:
news_data = []
for n in news_ds["newsdata"]:
  news_data.append({
    "id": n["article_id"],
    "title": n["title"],
    "source": n["source_name"],
    "date": n["pubDate"],
    "summary": n["description"]
  })

with open("news.json", "w") as ofp:
  json.dump(news_data, ofp)

In [ ]:
image_data = []
for n in news_ds["newsdata"]:
  img = n["image"]
  img = img.resize((img.width//2, img.height//2))
  colors = colorgram.extract(img, 4)

  image_data.append({
    "id": n["article_id"],
    "description": n["content"],
    "colors": {
      "proportions": [round(c.proportion, 3) for c in colors],
      "hex": [webcolors.rgb_to_hex(c.rgb) for c in colors],
      "rgb": [tuple(c.rgb) for c in colors],
    }
  })

with open("images.json", "w") as ofp:
  json.dump(image_data, ofp)

## Other Dataset

[https://huggingface.co/datasets/visualizedata/revolutionary_silhouettes](https://huggingface.co/datasets/visualizedata/revolutionary_silhouettes)

### Load and see what's in it

In [ ]:
sil_ds = load_dataset("visualizedata/revolutionary_silhouettes", split="original")
sil_ds

In [ ]:
# look at features
sil_ds.features

In [ ]:
# pick the first row/record
sil = sil_ds[0]
oimg = sil["image"]
display(oimg)

In [ ]:
cx,cy,cw,ch = sil["cx"], sil["cy"], sil["cw"], sil["ch"]
draw = PImageDraw.Draw(oimg)
draw.rectangle((cx, cy, cx+cw, cy+ch), outline=(200,0,0), width=4)
oimg

In [ ]:
xs = sorted([k for k in sil_ds[0].keys() if k.startswith("x")])
ys = sorted([k for k in sil_ds[0].keys() if k.startswith("y")])

draw = PImageDraw.Draw(oimg)
for kx, ky in zip(xs, ys):
  x,y = sil[kx], sil[ky]
  draw.circle((x,y), radius=2, fill=(200,0,0), outline=None)

oimg

### One more example

In [ ]:
from google.colab import userdata
HF_KEY = userdata.get("HF_KEY")

In [ ]:
art_ds = load_dataset("thiagohersan/mini-abstract", token=HF_KEY, split="train")
art_ds.features

In [ ]:
aoi = "kazimir-malevich"
km_idx = art_ds.features["label"].str2int(aoi)
km_idx

In [ ]:
km_ds = art_ds.filter(lambda r: r["label"] == km_idx, batch_size=16)

for a in km_ds.select((0,1,2)):
  display(a["image"])

## Exercise

Get accustomed to the Revolutionary Crossroads data and `Dataset`s.

### RevRoad `Dataset` Locations:

- https://huggingface.co/RevolutionCrossroads
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810 (247 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810_issues (707 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files (9.4 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files_PDFs (4.23 TB)
- https://huggingface.co/datasets/RevolutionCrossroads/si_us_revolutionary_era_collections (7.7 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_OCR_results/viewer/block_level (19.5 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_NER_Entity_Occurrences (939 MB)

In [ ]:
nara_ds = load_dataset("RevolutionCrossroads/nara_revolutionary_war_pension_files",
                       token=HF_KEY,
                       split="train",
                       streaming=True)
nara_ds.features

In [ ]:
nara0 = next(iter(nara_ds))

In [ ]:
nara010 = list(nara_ds.take(10))
nara010

In [ ]:
ocr_ds = load_dataset("RevolutionCrossroads/RevX_OCR_results",
                      # token=HF_KEY,
                      name="page_level",
                      split="train",
                      streaming=True)
ocr_ds.features

In [ ]:
ocr_items = list(ocr_ds.take(10)["new_ocr_text"])
ocr_items

In [ ]:
# Named Entity Recognition
ner_ds = load_dataset("RevolutionCrossroads/RevX_NER_Entity_Occurrences",
                      # token=HF_KEY,
                      name="all",
                      split="train",
                      streaming=True)
ner_ds.features